In [4]:
!pip install polars

   ---------------------------------------- 0.0/828.7 kB ? eta -:--:--
   ------------------------- -------------- 524.3/828.7 kB 4.6 MB/s eta 0:00:01
   ---------------------------------------- 828.7/828.7 kB 3.3 MB/s  0:00:00
   ---------------------------------------- 0.0/51.8 MB ? eta -:--:--
    --------------------------------------- 1.0/51.8 MB 5.0 MB/s eta 0:00:11
   - -------------------------------------- 2.1/51.8 MB 5.0 MB/s eta 0:00:11
   -- ------------------------------------- 3.4/51.8 MB 5.8 MB/s eta 0:00:09
   --- ------------------------------------ 4.5/51.8 MB 5.6 MB/s eta 0:00:09
   ---- ----------------------------------- 5.5/51.8 MB 5.5 MB/s eta 0:00:09
   ----- ---------------------------------- 6.8/51.8 MB 5.5 MB/s eta 0:00:09
   ------ --------------------------------- 8.1/51.8 MB 5.7 MB/s eta 0:00:08
   ------- -------------------------------- 9.4/51.8 MB 5.7 MB/s eta 0:00:08
   -------- ------------------------------- 10.7/51.8 MB 5.8 MB/s eta 0:00:08
   -----

In [7]:
import polars as pl

FILE_PATH = r"C:\Users\wolf2\anaconda_projects\NF-CSE-CIC-IDS2018-V2.parquet"

df = pl.read_parquet(FILE_PATH)

# 1. Форма датасета (строки, колонки)
print(" Форма датасета (строки, колонки):", df.shape)

# 2. Типы последних пяти колонок
print("\n Типы последних пяти колонок:")
print(df.dtypes[-5:])

# 3. Уникальные значения в колонке Label
print("\n Уникальные значения в Label:")
print(df["Label"].unique().sort().to_list())

# 4. Уникальные значения в колонке Attack (показываем первые 15, чтобы не захламлять вывод)
print("\n Уникальные значения в Attack (первые 15):")
attack_unique = df["Attack"].unique().sort()
print(attack_unique.head(15).to_list())
if len(attack_unique) > 15:
    print(f"   ... и ещё {len(attack_unique) - 15} типов")

# 5. Предварительный просмотр данных (как рекомендовано в подсказках)
print("\n Пример данных (первые 5 строк, выбранные колонки):")
print(df[["Label", "Attack", "PROTOCOL", "IN_BYTES", "OUT_BYTES", "FLOW_DURATION_MILLISECONDS"]].head(5))

# 6. Быстрая проверка на пропуски в ключевых колонках
print("\n Пропуски в ключевых колонках:")
for col in ["Label", "Attack", "PROTOCOL", "IN_BYTES", "OUT_BYTES", "FLOW_DURATION_MILLISECONDS"]:
    null_count = df[col].null_count()
    print(f"   {col}: {null_count:,} пропусков")

 Форма датасета (строки, колонки): (17129715, 43)

 Типы последних пяти колонок:
[Int16, Int32, Int8, Int8, String]

 Уникальные значения в Label:
[0, 1]

 Уникальные значения в Attack (первые 15):
['Benign', 'Bot', 'Brute Force -Web', 'Brute Force -XSS', 'DDOS attack-HOIC', 'DDOS attack-LOIC-UDP', 'DDoS attacks-LOIC-HTTP', 'DoS attacks-GoldenEye', 'DoS attacks-Hulk', 'DoS attacks-SlowHTTPTest', 'DoS attacks-Slowloris', 'FTP-BruteForce', 'Infilteration', 'SQL Injection', 'SSH-Bruteforce']

 Пример данных (первые 5 строк, выбранные колонки):
shape: (5, 6)
┌───────┬────────────────────────┬──────────┬──────────┬───────────┬────────────────────────────┐
│ Label ┆ Attack                 ┆ PROTOCOL ┆ IN_BYTES ┆ OUT_BYTES ┆ FLOW_DURATION_MILLISECONDS │
│ ---   ┆ ---                    ┆ ---      ┆ ---      ┆ ---       ┆ ---                        │
│ i8    ┆ str                    ┆ i8       ┆ i32      ┆ i32       ┆ i32                        │
╞═══════╪════════════════════════╪══════════╪══

In [8]:
# Подсчёт записей по меткам Label
n_benign = df.filter(pl.col("Label") == 0).shape[0]
n_attack = df.filter(pl.col("Label") != 0).shape[0]
total = df.shape[0]

print(f"Benign (Label == 0): {n_benign:,} записей ({n_benign/total*100:.2f}%)")
print(f"Attack (Label == 1): {n_attack:,} записей ({n_attack/total*100:.2f}%)")

Benign (Label == 0): 15,101,685 записей (88.16%)
Attack (Label == 1): 2,028,030 записей (11.84%)


In [9]:
# Добавляем колонку is_attack: 1 если атака, 0 если норма
df = df.with_columns(
    is_attack=(pl.col("Label") != 0).cast(pl.Int8)
)

# Проверка результата
print("Колонка is_attack добавлена. Пример значений:")
print(df.select(["Label", "is_attack"]).head(10))
print(f"\nСумма is_attack (всего атак): {df['is_attack'].sum():,}")

Колонка is_attack добавлена. Пример значений:
shape: (10, 2)
┌───────┬───────────┐
│ Label ┆ is_attack │
│ ---   ┆ ---       │
│ i8    ┆ i8        │
╞═══════╪═══════════╡
│ 1     ┆ 1         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 1     ┆ 1         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 0     ┆ 0         │
│ 1     ┆ 1         │
│ 0     ┆ 0         │
└───────┴───────────┘

Сумма is_attack (всего атак): 2,028,030


In [10]:
# Фильтруем только атаки и группируем по типу атаки
attack_agg = (
    df.filter(pl.col("Label") != 0)
    .group_by("Attack")
    .agg(
        avg_duration=pl.mean("FLOW_DURATION_MILLISECONDS"),
        avg_in_bytes=pl.mean("IN_BYTES"),
        count=pl.count()
    )
    .sort("avg_in_bytes", descending=True)
)

print("Агрегация по типам атак (отсортирована по avg_in_bytes):")
print(attack_agg)

# Сохранение результата в parquet
attack_agg.write_parquet("attack_summary_by_type.parquet")
print("\nФайл attack_summary_by_type.parquet успешно сохранён.")

C:\Users\wolf2\AppData\Local\Temp\ipykernel_8580\1721008570.py:8: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  count=pl.count()


Агрегация по типам атак (отсортирована по avg_in_bytes):
shape: (14, 4)
┌──────────────────────────┬───────────────┬──────────────┬─────────┐
│ Attack                   ┆ avg_duration  ┆ avg_in_bytes ┆ count   │
│ ---                      ┆ ---           ┆ ---          ┆ ---     │
│ str                      ┆ f64           ┆ f64          ┆ u32     │
╞══════════════════════════╪═══════════════╪══════════════╪═════════╡
│ DDOS attack-LOIC-UDP     ┆ 4.1959e6      ┆ 5.8540e6     ┆ 2112    │
│ DDoS attacks-LOIC-HTTP   ┆ 3.7241e6      ┆ 25991.904925 ┆ 207078  │
│ Brute Force -XSS         ┆ 3.7568e6      ┆ 16871.281553 ┆ 927     │
│ Brute Force -Web         ┆ 3.7553e6      ┆ 9797.653756  ┆ 2143    │
│ SSH-Bruteforce           ┆ 733291.768981 ┆ 5828.140747  ┆ 94979   │
│ …                        ┆ …             ┆ …            ┆ …       │
│ Infilteration            ┆ 234910.506514 ┆ 791.858613   ┆ 115513  │
│ Bot                      ┆ 1.2340e6      ┆ 677.354225   ┆ 28033   │
│ SQL Injection   

In [11]:
# Вывод топ-3 атак по среднему объёму входящего трафика
top3 = attack_agg.head(3)
print("Топ-3 типа атак по среднему объёму входящего трафика:")
print(top3)

Топ-3 типа атак по среднему объёму входящего трафика:
shape: (3, 4)
┌────────────────────────┬──────────────┬──────────────┬────────┐
│ Attack                 ┆ avg_duration ┆ avg_in_bytes ┆ count  │
│ ---                    ┆ ---          ┆ ---          ┆ ---    │
│ str                    ┆ f64          ┆ f64          ┆ u32    │
╞════════════════════════╪══════════════╪══════════════╪════════╡
│ DDOS attack-LOIC-UDP   ┆ 4.1959e6     ┆ 5.8540e6     ┆ 2112   │
│ DDoS attacks-LOIC-HTTP ┆ 3.7241e6     ┆ 25991.904925 ┆ 207078 │
│ Brute Force -XSS       ┆ 3.7568e6     ┆ 16871.281553 ┆ 927    │
└────────────────────────┴──────────────┴──────────────┴────────┘


In [12]:
# Общее распределение по PROTOCOL
print("Общее распределение по PROTOCOL:")
print(df["PROTOCOL"].value_counts().sort("count", descending=True))

# Распределение по PROTOCOL только для Benign
print("\nРаспределение по PROTOCOL только для Benign:")
print(df.filter(pl.col("Label") == 0)["PROTOCOL"].value_counts().sort("count", descending=True))

# Распределение по PROTOCOL только для Attack с разбивкой по Attack
print("\nРаспределение по PROTOCOL только для Attack (с разбивкой по Attack):")
print(df.filter(pl.col("Label") != 0)
      .group_by(["PROTOCOL", "Attack"])
      .agg(pl.count().alias("count"))
      .sort("count", descending=True))

# Сравнение PROTOCOL между Benign и Attack
print("\nСравнение распределения PROTOCOL между Benign и Attack:")
print(df.group_by(["PROTOCOL", "Label"])
       .agg(pl.count().alias("count"))
       .sort(["PROTOCOL", "Label"]))

Общее распределение по PROTOCOL:
shape: (6, 2)
┌──────────┬─────────┐
│ PROTOCOL ┆ count   │
│ ---      ┆ ---     │
│ i8       ┆ u32     │
╞══════════╪═════════╡
│ 6        ┆ 9346287 │
│ 17       ┆ 7776756 │
│ 1        ┆ 4857    │
│ 2        ┆ 976     │
│ 58       ┆ 836     │
│ 47       ┆ 3       │
└──────────┴─────────┘

Распределение по PROTOCOL только для Benign:
shape: (6, 2)
┌──────────┬─────────┐
│ PROTOCOL ┆ count   │
│ ---      ┆ ---     │
│ i8       ┆ u32     │
╞══════════╪═════════╡
│ 17       ┆ 7688529 │
│ 6        ┆ 7406897 │
│ 1        ┆ 4537    │
│ 2        ┆ 883     │
│ 58       ┆ 836     │
│ 47       ┆ 3       │
└──────────┴─────────┘

Распределение по PROTOCOL только для Attack (с разбивкой по Attack):


C:\Users\wolf2\AppData\Local\Temp\ipykernel_8580\4069983218.py:13: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("count"))


shape: (20, 3)
┌──────────┬────────────────────────┬─────────┐
│ PROTOCOL ┆ Attack                 ┆ count   │
│ ---      ┆ ---                    ┆ ---     │
│ i8       ┆ str                    ┆ u32     │
╞══════════╪════════════════════════╪═════════╡
│ 6        ┆ DDOS attack-HOIC       ┆ 1066881 │
│ 6        ┆ DoS attacks-Hulk       ┆ 432648  │
│ 6        ┆ DDoS attacks-LOIC-HTTP ┆ 189750  │
│ 6        ┆ SSH-Bruteforce         ┆ 94979   │
│ 17       ┆ Infilteration          ┆ 68676   │
│ …        ┆ …                      ┆ …       │
│ 6        ┆ SQL Injection          ┆ 432     │
│ 1        ┆ Infilteration          ┆ 320     │
│ 2        ┆ Infilteration          ┆ 93      │
│ 17       ┆ Brute Force -Web       ┆ 64      │
│ 17       ┆ Brute Force -XSS       ┆ 47      │
└──────────┴────────────────────────┴─────────┘

Сравнение распределения PROTOCOL между Benign и Attack:


C:\Users\wolf2\AppData\Local\Temp\ipykernel_8580\4069983218.py:19: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("count"))


shape: (10, 3)
┌──────────┬───────┬─────────┐
│ PROTOCOL ┆ Label ┆ count   │
│ ---      ┆ ---   ┆ ---     │
│ i8       ┆ i8    ┆ u32     │
╞══════════╪═══════╪═════════╡
│ 1        ┆ 0     ┆ 4537    │
│ 1        ┆ 1     ┆ 320     │
│ 2        ┆ 0     ┆ 883     │
│ 2        ┆ 1     ┆ 93      │
│ 6        ┆ 0     ┆ 7406897 │
│ 6        ┆ 1     ┆ 1939390 │
│ 17       ┆ 0     ┆ 7688529 │
│ 17       ┆ 1     ┆ 88227   │
│ 47       ┆ 0     ┆ 3       │
│ 58       ┆ 0     ┆ 836     │
└──────────┴───────┴─────────┘


In [13]:
# Группировка по is_attack и расчёт средних метрик
metrics_comp = (
    df.group_by("is_attack")
    .agg(
        avg_in_bytes=pl.mean("IN_BYTES"),
        avg_out_bytes=pl.mean("OUT_BYTES"),
        avg_duration=pl.mean("FLOW_DURATION_MILLISECONDS"),
        count=pl.count()
    )
    .sort("is_attack")
)

print("Сравнение средних метрик (0 = Benign, 1 = Attack):")
print(metrics_comp)

C:\Users\wolf2\AppData\Local\Temp\ipykernel_8580\349573498.py:8: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  count=pl.count()


Сравнение средних метрик (0 = Benign, 1 = Attack):
shape: (2, 5)
┌───────────┬──────────────┬───────────────┬───────────────┬──────────┐
│ is_attack ┆ avg_in_bytes ┆ avg_out_bytes ┆ avg_duration  ┆ count    │
│ ---       ┆ ---          ┆ ---           ┆ ---           ┆ ---      │
│ i8        ┆ f64          ┆ f64           ┆ f64           ┆ u32      │
╞═══════════╪══════════════╪═══════════════╪═══════════════╪══════════╡
│ 0         ┆ 867.640543   ┆ 8253.537161   ┆ 185715.810107 ┆ 15101685 │
│ 1         ┆ 10013.788006 ┆ 2301.412192   ┆ 3.7472e6      ┆ 2028030  │
└───────────┴──────────────┴───────────────┴───────────────┴──────────┘


In [14]:
# Создаём колонку is_suspicious по эвристическим правилам
# bytes_ratio = IN_BYTES / (OUT_BYTES + 1) — защита от деления на ноль
df = df.with_columns(
    bytes_ratio=pl.col("IN_BYTES") / (pl.col("OUT_BYTES") + 1)
)

df = df.with_columns(
    is_suspicious=(
        (pl.col("bytes_ratio") > 10) &
        (pl.col("FLOW_DURATION_MILLISECONDS") < 500) &
        (pl.col("IN_PKTS") > 10)
    ).cast(pl.Int8)
)

# Подсчёт результатов
total_flagged = df["is_suspicious"].sum()
true_attacks = df.filter(
    (pl.col("is_suspicious") == 1) & (pl.col("Label") != 0)
).shape[0]

accuracy = true_attacks / total_flagged if total_flagged > 0 else 0.0

print(f"Всего помечено как подозрительных (is_suspicious == 1): {total_flagged:,}")
print(f"Из них реальных атак (Label != 0): {true_attacks:,}")
print(f"Точность эвристики (True Positives / Total Flagged): {accuracy:.4f} ({accuracy*100:.2f}%)")

Всего помечено как подозрительных (is_suspicious == 1): 4,645
Из них реальных атак (Label != 0): 3,707
Точность эвристики (True Positives / Total Flagged): 0.7981 (79.81%)
